0. Install and import packages

In [10]:
# If needed:
!pip install transformers torch pandas numpy scipy scikit-learn tqdm

import os
import re
import math
import string
import itertools
import numpy as np
import pandas as pd
import torch

from tqdm import tqdm
from collections import defaultdict

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

1. Configuration

In [2]:
# --------------------------------------------------
# User configuration
# --------------------------------------------------

INPUT_PATH = "funding_organization_frequency.csv"

# Choose either BERT or SciBERT deployment
MODEL_TYPE = "BERT"   # Options: "BERT" or "SciBERT"

# Replace this with the actual saved model path from your experiment
BEST_MODEL_PATH = "./saved_models/bert_best_fold"

print("Model path exists:", os.path.exists(BEST_MODEL_PATH))

if os.path.exists(BEST_MODEL_PATH):
    print("Files:", os.listdir(BEST_MODEL_PATH)[:10])
else:
    raise FileNotFoundError(
        f"BEST_MODEL_PATH does not exist: {BEST_MODEL_PATH}. "
        "Please update BEST_MODEL_PATH to the actual saved model directory."
    )

# HAC distance threshold selected from validation
# BERT: 0.10/0.25
# SciBERT: 0.10
HAC_DISTANCE_THRESHOLD = 0.10

# BERT/SciBERT inference settings
MAX_LENGTH = 64
BATCH_SIZE = 256

# Blocking settings
MIN_BLOCK_SIZE = 2
MAX_BLOCK_SIZE = 500

# Output files
CANDIDATE_PAIR_OUTPUT = "deployment_candidate_pairs.csv"
PAIRWISE_SCORE_OUTPUT = "deployment_pairwise_scores.csv"
FINAL_CLUSTER_OUTPUT = "deployment_final_ond_clusters.csv"

# Device setting
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

Model path exists: True
Files: ['config.json', 'trainer_state.json', 'model.safetensors', 'optimizer.pt', 'rng_state.pth', 'training_args.bin', 'scheduler.pt']
Using device: cuda
CUDA available: True
Number of GPUs: 4
0 NVIDIA A10G
1 NVIDIA A10G
2 NVIDIA A10G
3 NVIDIA A10G


2. Load deployment data

In [3]:
# --------------------------------------------------
# Load funding organization frequency data
# --------------------------------------------------

df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")

print(df.head())
print(df.columns)
print("Number of rows:", len(df))

# --------------------------------------------------
# Standardize column names if needed
# --------------------------------------------------

# Expected columns:
# Instance_id, Funding_Organization, Frequency
# If your file uses different names, modify this mapping.

column_map = {
    "InstanceID": "Instance_id",
    "Instance_ID": "Instance_id",
    "OrgName": "Funding_Organization",
    "Organization": "Funding_Organization",
    "Count": "Frequency",
    "Freq": "Frequency"
}

df = df.rename(columns={c: column_map[c] for c in df.columns if c in column_map})

required_cols = ["Funding_Organization", "Frequency"]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# --------------------------------------------------
# Create Instance_id if it does not exist
# --------------------------------------------------

if "Instance_id" not in df.columns:
    df["Instance_id"] = [
        f"DEPLOY_{i:08d}" for i in range(1, len(df) + 1)
    ]

# Make sure ID and text columns are strings
df["Instance_id"] = df["Instance_id"].astype(str)
df["Funding_Organization"] = df["Funding_Organization"].astype(str)

# Make sure frequency is numeric
df["Frequency"] = pd.to_numeric(df["Frequency"], errors="coerce").fillna(0).astype(int)

# Remove empty organization names
df = df[df["Funding_Organization"].str.strip() != ""].copy()
df = df.reset_index(drop=True)

print("Number of valid organization names:", len(df))
df.head()

   Instance_id                               Funding_Organization  Frequency
0      1000001       National Natural Science Foundation of China       6580
1      1000002                        National Science Foundation       3743
2      1000003                      National Institutes of Health       1594
3      1000004                   Horizon 2020 Framework Programme       1432
4      1000005  National Office for Philosophy and Social Scie...       1027
Index(['Instance_id', 'Funding_Organization', 'Frequency'], dtype='object')
Number of rows: 21091
Number of valid organization names: 21091


,Instance_id,Funding_Organization,Frequency
0,1000001,National Natural Science Foundation of China,6580
1,1000002,National Science Foundation,3743
2,1000003,National Institutes of Health,1594
3,1000004,Horizon 2020 Framework Programme,1432
4,1000005,National Office for Philosophy and Social Scie...,1027


3. Blocking functions

In [4]:
# --------------------------------------------------
# Text normalization
# --------------------------------------------------

GENERIC_TERMS = {
    "university", "institute", "institution", "college", "school",
    "center", "centre", "department", "division", "office",
    "foundation", "fund", "funding", "agency", "council",
    "research", "science", "scientific", "technology",
    "ministry", "national", "international", "program", "programme",
    "project", "grant", "association", "society", "organization",
    "organisation", "government", "federal", "state"
}

def normalize_text(text):
    # Lowercase text
    text = str(text).lower()
    
    # Replace punctuation with spaces
    text = text.translate(str.maketrans(string.punctuation, " " * len(string.punctuation)))
    
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def tokenize_org_name(text):
    # Tokenize normalized organization name
    text = normalize_text(text)
    tokens = text.split()
    return tokens


def get_first_token(text):
    # Return the first token
    tokens = tokenize_org_name(text)
    return tokens[0] if tokens else None


def get_last_token(text):
    # Return the last token
    tokens = tokenize_org_name(text)
    return tokens[-1] if tokens else None


def get_important_tokens(text):
    # Return non-generic tokens
    tokens = tokenize_org_name(text)
    important_tokens = [
        t for t in tokens
        if t not in GENERIC_TERMS and len(t) > 2
    ]
    return important_tokens


def get_acronym(text):
    # Build acronym from alphabetic tokens
    tokens = tokenize_org_name(text)
    acronym = "".join([t[0].upper() for t in tokens if t and t[0].isalpha()])
    return acronym if len(acronym) >= 2 else None


def get_blocking_keys(text):
    # Generate multiple blocking keys for one organization name
    keys = set()
    
    first_token = get_first_token(text)
    last_token = get_last_token(text)
    acronym = get_acronym(text)
    important_tokens = get_important_tokens(text)
    
    if first_token:
        keys.add(f"FIRST::{first_token}")
    
    if last_token:
        keys.add(f"LAST::{last_token}")
    
    if acronym:
        keys.add(f"ACRONYM::{acronym}")
    
    for token in important_tokens[:3]:
        keys.add(f"IMPORTANT::{token}")
    
    return keys

4. Generate candidate pairs using blocking

In [5]:
# --------------------------------------------------
# Build blocks
# --------------------------------------------------

blocks = defaultdict(list)

for _, row in tqdm(df.iterrows(), total=len(df), desc="Building blocks"):
    instance_id = row["Instance_id"]
    org_name = row["Funding_Organization"]
    
    keys = get_blocking_keys(org_name)
    
    for key in keys:
        blocks[key].append(instance_id)

print("Number of raw blocks:", len(blocks))

# --------------------------------------------------
# Filter block sizes
# --------------------------------------------------

filtered_blocks = {
    key: ids
    for key, ids in blocks.items()
    if MIN_BLOCK_SIZE <= len(ids) <= MAX_BLOCK_SIZE
}

print("Number of filtered blocks:", len(filtered_blocks))

# --------------------------------------------------
# Generate unique candidate pairs from blocks
# --------------------------------------------------

candidate_pair_set = set()

for key, ids in tqdm(filtered_blocks.items(), desc="Generating candidate pairs"):
    unique_ids = sorted(set(ids))
    
    for id1, id2 in itertools.combinations(unique_ids, 2):
        pair_key = tuple(sorted([id1, id2]))
        candidate_pair_set.add(pair_key)

print("Number of candidate pairs:", len(candidate_pair_set))

# --------------------------------------------------
# Convert candidate pairs to DataFrame
# --------------------------------------------------

id_to_name = dict(zip(df["Instance_id"], df["Funding_Organization"]))

candidate_rows = []

for id1, id2 in candidate_pair_set:
    candidate_rows.append({
        "InstanceID_1": id1,
        "OrgName_1": id_to_name[id1],
        "InstanceID_2": id2,
        "OrgName_2": id_to_name[id2]
    })

candidate_pairs_df = pd.DataFrame(candidate_rows)

candidate_pairs_df.to_csv(
    CANDIDATE_PAIR_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print(candidate_pairs_df.head())
print("Saved:", CANDIDATE_PAIR_OUTPUT)

Building blocks: 100%|██████████| 21091/21091 [00:01<00:00, 12181.61it/s]


Number of raw blocks: 38595
Number of filtered blocks: 9056


Generating candidate pairs: 100%|██████████| 9056/9056 [00:01<00:00, 5075.23it/s]


Number of candidate pairs: 2531119
  InstanceID_1                                          OrgName_1  \
0      1000044                   Agence Nationale de la Recherche   
1      1000831          University of Chinese Academy of Sciences   
2      1014148  Center of Innovative Fintech Business Models, ...   
3      1007410                   Visiting Advanced Joint Research   
4      1007184  Key Projects of Shanghai Soft Science Research...   

  InstanceID_2                                          OrgName_2  
0      1005132       Association de Recherche sur la Polyarthrite  
1      1018690  Central Public-interest Scientific Institution...  
2      1015895                         Bureau of Industry, Taiwan  
3      1015723                      Icelandic Centre for Research  
4      1016912              Key R &D Program of Zhejiang Province  
Saved: deployment_candidate_pairs.csv


5. Run pairwise classification deployment with Terminal

### Bash_1

cd ~/SageMaker
ls

### Bash_2

ls run_multigpu_inference.py

### Bash_3

pip install transformers pandas numpy tqdm scikit-learn scipy

### Bash_4

python run_multigpu_inference.py \
  --candidate_pairs deployment_candidate_pairs.csv \
  --model_path ./saved_models/bert_best_fold \
  --model_type BERT \
  --batch_size 256 \
  --max_length 64 \
  --num_gpus 4 \
  --output_dir deployment_pairwise_score_shards \
  --combined_output deployment_pairwise_scores.csv


In [6]:
pairwise_scores_df = pd.read_csv(
    "deployment_pairwise_scores.csv",
    encoding="utf-8-sig"
)

7. Build similarity matrix for HAC clustering

In [7]:
# --------------------------------------------------
# Build similarity matrix
# --------------------------------------------------

def build_similarity_matrix(all_instances_df, scores_df):
    instance_ids = all_instances_df["Instance_id"].astype(str).tolist()
    id_to_idx = {instance_id: i for i, instance_id in enumerate(instance_ids)}
    
    n = len(instance_ids)
    sim_matrix = np.zeros((n, n), dtype=np.float32)
    
    # Self-similarity is 1
    np.fill_diagonal(sim_matrix, 1.0)
    
    for _, row in tqdm(scores_df.iterrows(), total=len(scores_df), desc="Building similarity matrix"):
        id1 = str(row["InstanceID_1"])
        id2 = str(row["InstanceID_2"])
        score = float(row["SimilarityScore"])
        
        if id1 in id_to_idx and id2 in id_to_idx:
            i = id_to_idx[id1]
            j = id_to_idx[id2]
            sim_matrix[i, j] = score
            sim_matrix[j, i] = score
    
    return instance_ids, sim_matrix


instance_ids, sim_matrix = build_similarity_matrix(df, pairwise_scores_df)

print("Similarity matrix shape:", sim_matrix.shape)

Building similarity matrix: 100%|██████████| 2531119/2531119 [01:56<00:00, 21782.27it/s]


Similarity matrix shape: (21091, 21091)


8. Run HAC clustering

In [11]:
# --------------------------------------------------
# Run HAC clustering
# --------------------------------------------------

def run_hac_clustering(sim_matrix, instance_ids, threshold, linkage_method="average"):
    # Convert similarity to distance
    distance_matrix = 1.0 - sim_matrix
    
    # Ensure diagonal distance is zero
    np.fill_diagonal(distance_matrix, 0.0)
    
    # Convert square distance matrix to condensed distance format
    condensed_distance = squareform(distance_matrix, checks=False)
    
    # Run hierarchical agglomerative clustering
    Z = linkage(condensed_distance, method=linkage_method)
    
    # Cut dendrogram at selected distance threshold
    cluster_labels = fcluster(
        Z,
        t=threshold,
        criterion="distance"
    )
    
    cluster_df = pd.DataFrame({
        "Instance_id": instance_ids,
        "cluster_label_raw": cluster_labels
    })
    
    return cluster_df


cluster_assignments_df = run_hac_clustering(
    sim_matrix=sim_matrix,
    instance_ids=instance_ids,
    threshold=HAC_DISTANCE_THRESHOLD,
    linkage_method="average"
)

# Create clean deployment cluster IDs
cluster_assignments_df["cluster_id"] = (
    cluster_assignments_df["cluster_label_raw"]
    .rank(method="dense")
    .astype(int)
    .apply(lambda x: f"OND_CLUSTER_{x:08d}")
)

print(cluster_assignments_df.head())
print("Number of clusters:", cluster_assignments_df["cluster_id"].nunique())

  Instance_id  cluster_label_raw            cluster_id
0     1000001               8426  OND_CLUSTER_00008426
1     1000002               3958  OND_CLUSTER_00003958
2     1000003               6140  OND_CLUSTER_00006140
3     1000004               3723  OND_CLUSTER_00003723
4     1000005               5065  OND_CLUSTER_00005065
Number of clusters: 10604


9. Assign representative organization name by highest frequency

In [12]:
# --------------------------------------------------
# Merge cluster assignments with original deployment data
# --------------------------------------------------

deployment_result_df = df.merge(
    cluster_assignments_df[["Instance_id", "cluster_id"]],
    on="Instance_id",
    how="left"
)

# --------------------------------------------------
# Select representative name for each cluster
# The representative name is the Funding_Organization with the highest frequency in the cluster.
# If there is a tie, the shorter name is selected.
# --------------------------------------------------

def select_representative_name(group):
    temp = group.copy()
    
    temp["NameLength"] = temp["Funding_Organization"].astype(str).str.len()
    
    temp = temp.sort_values(
        by=["Frequency", "NameLength", "Funding_Organization"],
        ascending=[False, True, True]
    )
    
    representative_name = temp.iloc[0]["Funding_Organization"]
    representative_instance_id = temp.iloc[0]["Instance_id"]
    representative_frequency = temp.iloc[0]["Frequency"]
    
    return pd.Series({
        "Representative_Organization_Name": representative_name,
        "Representative_Instance_id": representative_instance_id,
        "Representative_Frequency": representative_frequency
    })


representatives_df = (
    deployment_result_df
    .groupby("cluster_id", as_index=False)
    .apply(select_representative_name)
    .reset_index(drop=True)
)

print(representatives_df.head())

# --------------------------------------------------
# Merge representative names back to all instances
# --------------------------------------------------

deployment_result_df = deployment_result_df.merge(
    representatives_df,
    on="cluster_id",
    how="left"
)

deployment_result_df.head()

             cluster_id Representative_Organization_Name  \
0  OND_CLUSTER_00000001   Tiroler Wissenschaftsförderung   
1  OND_CLUSTER_00000002                        TJU-Wenge   
2  OND_CLUSTER_00000003                         McKenzie   
3  OND_CLUSTER_00000004         Florence Violet McKenzie   
4  OND_CLUSTER_00000005                         Ian Kerr   

  Representative_Instance_id  Representative_Frequency  
0                    1005312                         2  
1                    1013286                         1  
2                    1007149                         1  
3                    1013416                         1  
4                    1016918                         1  


/tmp/ipykernel_16858/3346929655.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_representative_name)


,Instance_id,Funding_Organization,Frequency,cluster_id,Representative_Organization_Name,Representative_Instance_id,Representative_Frequency
0,1000001,National Natural Science Foundation of China,6580,OND_CLUSTER_00008426,National Natural Science Foundation of China,1000001,6580
1,1000002,National Science Foundation,3743,OND_CLUSTER_00003958,National Science Foundation,1000002,3743
2,1000003,National Institutes of Health,1594,OND_CLUSTER_00006140,National Institutes of Health,1000003,1594
3,1000004,Horizon 2020 Framework Programme,1432,OND_CLUSTER_00003723,Horizon 2020 Framework Programme,1000004,1432
4,1000005,National Office for Philosophy and Social Scie...,1027,OND_CLUSTER_00005065,National Office for Philosophy and Social Scie...,1000005,1027


10. Save final OND deployment file

In [13]:
# --------------------------------------------------
# Reorder output columns
# --------------------------------------------------

output_cols = [
    "Instance_id",
    "Funding_Organization",
    "Frequency",
    "cluster_id",
    "Representative_Organization_Name",
    "Representative_Instance_id",
    "Representative_Frequency"
]

# Keep any additional original columns at the end
additional_cols = [
    col for col in deployment_result_df.columns
    if col not in output_cols
]

deployment_result_df = deployment_result_df[output_cols + additional_cols]

# --------------------------------------------------
# Save final deployment result
# --------------------------------------------------

deployment_result_df.to_csv(
    FINAL_CLUSTER_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", FINAL_CLUSTER_OUTPUT)
print("Number of instances:", len(deployment_result_df))
print("Number of clusters:", deployment_result_df["cluster_id"].nunique())

deployment_result_df.head()

Saved: deployment_final_ond_clusters.csv
Number of instances: 21091
Number of clusters: 10604


,Instance_id,Funding_Organization,Frequency,cluster_id,Representative_Organization_Name,Representative_Instance_id,Representative_Frequency
0,1000001,National Natural Science Foundation of China,6580,OND_CLUSTER_00008426,National Natural Science Foundation of China,1000001,6580
1,1000002,National Science Foundation,3743,OND_CLUSTER_00003958,National Science Foundation,1000002,3743
2,1000003,National Institutes of Health,1594,OND_CLUSTER_00006140,National Institutes of Health,1000003,1594
3,1000004,Horizon 2020 Framework Programme,1432,OND_CLUSTER_00003723,Horizon 2020 Framework Programme,1000004,1432
4,1000005,National Office for Philosophy and Social Scie...,1027,OND_CLUSTER_00005065,National Office for Philosophy and Social Scie...,1000005,1027


11. Optional quality check

In [14]:
# --------------------------------------------------
# Inspect largest clusters
# --------------------------------------------------

cluster_size_df = (
    deployment_result_df
    .groupby("cluster_id")
    .agg(
        ClusterSize=("Instance_id", "count"),
        TotalFrequency=("Frequency", "sum"),
        Representative_Organization_Name=("Representative_Organization_Name", "first")
    )
    .reset_index()
    .sort_values(["ClusterSize", "TotalFrequency"], ascending=False)
)

cluster_size_df.to_csv(
    "deployment_cluster_size_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print(cluster_size_df.head(20))

                cluster_id  ClusterSize  TotalFrequency  \
5919  OND_CLUSTER_00005920           75             137   
4557  OND_CLUSTER_00004558           72             162   
3346  OND_CLUSTER_00003347           65              82   
8833  OND_CLUSTER_00008834           59              76   
8224  OND_CLUSTER_00008225           55              92   
5555  OND_CLUSTER_00005556           55              66   
9292  OND_CLUSTER_00009293           55              57   
8220  OND_CLUSTER_00008221           53              58   
3074  OND_CLUSTER_00003075           50              64   
3344  OND_CLUSTER_00003345           48              50   
6820  OND_CLUSTER_00006821           45              59   
8602  OND_CLUSTER_00008603           44              48   
3063  OND_CLUSTER_00003064           42              48   
5550  OND_CLUSTER_00005551           39              70   
9289  OND_CLUSTER_00009290           38             105   
9286  OND_CLUSTER_00009287           38              53 

In [13]:
# --------------------------------------------------
# Inspect names in a selected cluster
# --------------------------------------------------

def inspect_cluster(cluster_id):
    temp = deployment_result_df[
        deployment_result_df["cluster_id"] == cluster_id
    ].copy()
    
    temp = temp.sort_values("Frequency", ascending=False)
    
    return temp[
        [
            "Instance_id",
            "Funding_Organization",
            "Frequency",
            "cluster_id",
            "Representative_Organization_Name"
        ]
    ]


# Example:
# inspect_cluster("OND_CLUSTER_00000001")